# Tutorial 3: Forward and Backward Model Selection
## Compare predictor sets without allowing the test set to choose the model

**Course:** IE 1171  
**Input file:** `tutorial_1_output.csv`  
**Level 1:** Required core—forward and backward selection  
**Level 2:** Optional deep dives—stability, information criteria, shrinkage, grouped features, and social auditing

---

Tutorial 2 fit a multiple linear regression using a prespecified set of predictors. Tutorial 3 asks a different question:

> Which predictor set should be carried forward when several candidate models are possible?

Claude will draft the selection code. Your job is to define the criterion, prevent data leakage, compare procedures, inspect stability, and decide whether a smaller model is actually preferable.


# <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="44" style="vertical-align:middle; margin-right:10px;"> Learning Objectives

By the end of this tutorial, you should be able to:

1. **Understand why models are selected**
   - Distinguish null, full, and subset models.
   - Explain the complexity, interpretation, and generalization tradeoffs.
   - Describe how forward and backward selection search through predictors.
2. **Compare models honestly**
   - Use training-only cross-validation to guide selection.
   - Keep the final test rows untouched until the candidate models are chosen.
   - Compare null, full, and selected models with the same evaluation measures.
3. **Judge what a selected model means**
   - Examine whether selected features are stable.
   - Contrast stepwise selection with ridge and lasso.
   - Avoid treating selected variables as automatically causal, important, or fair across groups.


# Pólya’s Four-Step Problem-Solving Cycle

> **Backbone for this tutorial:** George Pólya’s four steps organize the work from problem framing through verification. The steps are a **cycle**, not a one-way checklist: if later evidence exposes a bad assumption, return to the earlier step that needs revision.

| Marker | Pólya step | Guiding question | In this tutorial |
|---|---|---|---|
| **🔵 🧭** | **Understand the Problem** | What is the real problem, what is known, and what constraints define success? | Define the model-selection goal, candidate predictors, and what “better” will mean. |
| **🟣 🗺️** | **Devise a Plan** | What sequence of actions and checks should connect the current state to the goal? | Lock the train/test split and training-only selection criterion before searching among models. |
| **🟠 🛠️** | **Carry Out the Plan** | Can the plan be executed in small, observable steps and checked as it runs? | Execute forward and backward selection without using the test set to make selection decisions. |
| **🟢 🔎** | **Look Back** | Does the result answer the original problem, and what should be revised or generalized? | Compare the paths, final models, test evidence, and stability before choosing a conclusion. |

The colored markers reappear at the points where each step becomes the main focus. **Human Checks support the cycle, but they are not a universal checklist:** meaningful verification depends on domain knowledge, the data-generating process, and the consequences of being wrong.


## <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="36" style="vertical-align:middle; margin-right:9px;"> Assigned Reading

## <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="36" style="vertical-align:middle; margin-right:9px;"> Statistical Reading Used Throughout Level 1 and Level 2

**James et al., _An Introduction to Statistical Learning with Applications in Python_ (ISLP)**

- Section 6.1: Subset Selection
- Section 6.2: Shrinkage Methods

Focus on:

- best subset selection;
- forward stepwise selection;
- backward stepwise selection;
- training error versus test error;
- cross-validation;
- $C_p$, AIC, BIC, and adjusted $R^2$;
- the difference between choosing a discrete subset and shrinking coefficients;
- ridge regression and lasso regression.

## Ethical / Social-Good Reading

**Michael Kearns and Aaron Roth, _The Ethical Algorithm_**

- **“Smoking May Be Harmful to Your Privacy”** (pp. 34–36)
- **“A Different(ial) Notion of Privacy”** (pp. 36–39)

Use these readings to examine whether feature selection reduces sensitive-data exposure, leaves privacy-revealing proxies, or changes what can be inferred about people.


## Reading Questions

Answer before coding:

1. Why does training error almost always improve when predictors are added?
2. Why is the model with the smallest training error not automatically the best model?
3. How do forward and backward selection search different paths?
4. Why can both procedures miss the globally best subset?
5. What criterion will determine whether a feature is added or removed here?
6. Why must the test set remain outside the selection process?
7. How do ridge and lasso differ from stepwise selection?
8. Could a simpler model be less fair even if its overall test RMSE is similar?


# <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="44" style="vertical-align:middle; margin-right:10px;"> Tutorial Flow

Each programming section uses the same pattern:

| Section | Purpose |
|---|---|
| **Concept and reading connection** | Define what the procedure is trying to accomplish. |
| **Without Claude** | Identify the details that make implementation difficult. |
| **Claude Coding Task** | Request one focused code cell. |
| **Your Workspace** | Read and run Claude’s response. |
| **Reference Solution** | Compare after attempting the task. |
| **Human Check** | Verify leakage, criterion, feature path, and output. |
| **Look Back** | Decide whether the result is stable, useful, and responsible. |

## Non-negotiable rule

> The test set may evaluate the final candidate models. It may not decide which feature is added, removed, tuned, or retained.


## Tutorial Symbols

| Symbol | Meaning | What to do |
|---|---|---|
| **🔵 🧭  🟣 🗺️  🟠 🛠️  🟢 🔎** | **Pólya Backbone** | Treat the four colored checkpoints as the main problem-solving cycle; return to an earlier step when new evidence requires revision. |
| <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="28" style="vertical-align:middle; margin-right:8px;"> | **Tutorial Flow** | Follow the notebook’s normal route. |
| <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="28" style="vertical-align:middle; margin-right:8px;"> | **Learning Objectives** | See the three destinations for the tutorial. |
| <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="28" style="vertical-align:middle; margin-right:8px;"> | **Assigned Reading** | Read the named sections before or alongside the notebook. |
| <img src="tutorial-icons/theory.png" alt="Theory" width="28" style="vertical-align:middle; margin-right:8px;"> | **Theory** | Connect the assigned reading to the current part. |
| <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="28" style="vertical-align:middle; margin-right:8px;"> | **Manual Pause** | Think or predict before asking Claude. |
| <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="28" style="vertical-align:middle; margin-right:8px;"> | **Without Claude** | Notice the programming details Claude can coordinate. |
| <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="28" style="vertical-align:middle; margin-right:8px;"> | **Claude Task** | Use one focused and checkable prompt. |
| <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="28" style="vertical-align:middle; margin-right:8px;"> | **Your Workspace** | Paste, read, and run Claude’s response. |
| <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="28" style="vertical-align:middle; margin-right:8px;"> | **Reference Solution** | Compare only after your own attempt. |
| <img src="tutorial-icons/human_check.png" alt="Human Check" width="28" style="vertical-align:middle; margin-right:8px;"> | **Human Check — domain expertise required** | Use the provided questions, then add a domain-specific check. The notebook cannot supply a complete checklist for every application. |
| <img src="tutorial-icons/look_back.png" alt="Look Back" width="28" style="vertical-align:middle; margin-right:8px;"> | **Look Back** | Interpret, challenge, and reflect on the result. |
| <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="28" style="vertical-align:middle; margin-right:8px;"> | **Level 2 — Challenge Ahead** | Take the optional harder route after Level 1. |

> **Important — Human Checks are not a complete checklist.** The notebook can suggest generic verification questions, but deciding what *must* be checked depends on knowledge of the domain, the data-generating process, and the consequences of an error. If you do not have that expertise, involve someone who does. Every Human Check asks you to add your own domain-specific question.

# <img src="tutorial-icons/theory.png" alt="Theory" width="44" style="vertical-align:middle; margin-right:10px;"> Theory Foundation: Selecting a Model Without Selecting on the Test Set

With $p$ candidate predictors there are $2^p$ possible subsets. Exhaustive search becomes expensive quickly, so forward and backward selection use greedy paths. Forward selection starts with no predictors and repeatedly adds the candidate that most improves a training-only criterion. Backward selection starts with all eligible predictors and removes one at a time. Neither method guarantees the globally best subset, and the two paths can disagree when predictors carry overlapping information.

Selection must estimate **generalization**, not reward a model for fitting the rows used to choose it. In $K$-fold cross-validation, training data are partitioned into folds $F_1,\ldots,F_K$ and the validation estimate is

$$
CV_K=\frac{1}{K}\sum_{k=1}^{K}
\frac{1}{|F_k|}\sum_{i\in F_k}L\!\left(y_i,\hat f^{(-k)}(x_i)\right),
$$

where $\hat f^{(-k)}$ is fitted without fold $k$. If preprocessing or feature selection is performed before this loop, validation information leaks into training. The entire selection procedure belongs inside each training fold.

Repeatedly comparing models also creates **selection optimism**: the winning cross-validation score is partly the best observed noise among many candidates. This is why the untouched test set is used once, after the candidate and hyperparameters have been fixed. The test result estimates the performance of the full selection pipeline—not the eternal quality of a predictor set.

Subset selection is only one approach. Ridge regression minimizes $RSS+\lambda\sum_j\beta_j^2$ and keeps all predictors while shrinking coefficients. Lasso minimizes $RSS+\lambda\sum_j|\beta_j|$ and can set some coefficients to zero. These methods trade bias for reduced variance and require scaling when predictor units differ.

### Questions you should be ready to answer

- Why are there $2^p$ subsets, and why do greedy methods examine fewer?
- What information is allowed during selection and what must remain locked?
- Why can forward and backward procedures return different models?
- Why does a selected variable not automatically have a causal interpretation?

## 🔵 🧭 Pólya Step 1 — Understand the Problem

**Backbone checkpoint.** State the real goal, evidence, constraints, and what would count as success before asking an AI system to solve anything.

**In this tutorial:** Define the model-selection goal, candidate predictors, and what “better” will mean.

## Level 1 — Required Core

### Part 0: Define the Model-Selection Problem

The response remains:

$$
Y = \texttt{PROPr}
$$

To keep the required tutorial understandable, Level 1 considers seven numeric candidate predictors:

- `StressBurden`
- `SupportFrequency`
- `FinancialDifficultyBurden`
- `FoodInsecurity`
- `ChronicConditionCount`
- `AGE`
- `BMI`

`INCOME` and `EDUC4` are retained for later auditing but are not candidates in the required stepwise search. Treating categorical variables correctly requires groups of indicator columns; that issue appears in the Level 2 grouped-feature deep dive.

#### Candidate models

- **Null model:** predicts the training mean and uses no substantive predictors.
- **Full model:** uses all seven candidate predictors.
- **Subset model:** uses some, but not all, candidate predictors.


#### <img src="tutorial-icons/theory.png" alt="Theory" width="26" style="vertical-align:middle; margin-right:8px;"> Theory: What Does “Best” Mean?

The word **best** is incomplete unless a criterion is stated.

Possible criteria include:

- training residual sum of squares;
- adjusted $R^2$;
- Mallows’ $C_p$;
- AIC;
- BIC;
- validation error;
- cross-validation error;
- final test error;
- interpretability, cost, stability, or fairness.

This tutorial uses:

> **Mean five-fold cross-validated RMSE on the training set**

A feature is added or removed only when it improves that criterion by more than a small tolerance.

The untouched test set is used once, after the candidate models have been selected.

### Deeper explanation

“Best” requires a loss, validation design, and complexity rule. Minimizing training RSS always weakly favors adding predictors, so it cannot by itself choose subset size. Adjusted $R^2$, AIC, BIC, and cross-validation penalize complexity in different ways and target different goals. A small average error difference may be dominated by fold-to-fold variation, so stability and interpretability should be considered alongside the winning mean score.


### Part 1: Load the Tutorial 1 Output

#### <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="26" style="vertical-align:middle; margin-right:8px;"> Without Claude

You would need to remember file checks, imports, expected columns, missingness inspection, and the distinction between the file name and the in-memory DataFrame.

#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 1

```text
Act as a careful Python tutor.

Write one Jupyter Notebook code cell that:
1. imports Path from pathlib;
2. imports numpy as np, pandas as pd, and matplotlib.pyplot as plt;
3. imports display from IPython.display;
4. checks that tutorial_1_output.csv exists;
5. raises a clear FileNotFoundError if it is missing;
6. loads the file into a DataFrame named tutorial_1_output;
7. prints the shape and column names;
8. displays the first five rows;
9. displays missing counts and percentages;
10. confirms that PROPr and the seven numeric candidate predictors exist.

Do not modify the data.
Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 1: Load and inspect the handoff file
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 1


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

DATA_FILE = Path("tutorial_1_output.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Run Tutorial 1 first so that tutorial_1_output.csv is created."
    )

tutorial_1_output = pd.read_csv(DATA_FILE)

candidate_predictors = [
    "StressBurden",
    "SupportFrequency",
    "FinancialDifficultyBurden",
    "FoodInsecurity",
    "ChronicConditionCount",
    "AGE",
    "BMI",
]

required_columns = ["PROPr", *candidate_predictors]

missing_columns = [
    column for column in required_columns
    if column not in tutorial_1_output.columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

print("Loaded shape:", tutorial_1_output.shape)
print("\nColumns:")
print(tutorial_1_output.columns.tolist())

display(tutorial_1_output.head())

missing_table = (
    tutorial_1_output
    .isna()
    .sum()
    .to_frame("missing_count")
    .assign(
        missing_percent=lambda table:
        100 * table["missing_count"] / len(tutorial_1_output)
    )
    .sort_values("missing_percent", ascending=False)
)

print("\nMissing-value summary:")
display(missing_table)


#### <img src="tutorial-icons/human_check.png" alt="Human Check" width="26" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Does the file come from Tutorial 1?
- Are all candidate predictors present?
- Are `INCOME` and `EDUC4` available for later auditing?
- Which rows may be excluded when the modeling table is created?

## 🟣 🗺️ Pólya Step 2 — Devise a Plan

**Backbone checkpoint.** Decide the sequence of actions and checks before the main execution. Make assumptions, evaluation rules, and stopping conditions visible so they can be challenged.

**In this tutorial:** Lock the train/test split and training-only selection criterion before searching among models.

### Part 2: Create One Training/Test Split

Model selection repeatedly examines candidate models. Therefore:

- the **training set** supports selection;
- cross-validation occurs only inside the training set;
- the **test set** remains untouched until final comparison.

#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 2

```text
Act as a careful Python tutor.

I have a DataFrame named tutorial_1_output and a list named
candidate_predictors.

Write one Jupyter Notebook code cell that:
1. sets target equal to PROPr;
2. creates model_df containing PROPr, the candidate predictors,
   INCOME, and EDUC4;
3. drops rows only when PROPr or a candidate predictor is missing;
4. prints the number of rows retained and removed;
5. creates X from the candidate predictors and y from PROPr;
6. imports train_test_split;
7. creates one 80/20 split using random_state=1099;
8. preserves original row indices;
9. prints training and test sizes;
10. asserts that training and test indices do not overlap.

Do not perform feature selection yet.
Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 2: Create the locked test split
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 2


In [ ]:
from sklearn.model_selection import train_test_split

target = "PROPr"

model_df = tutorial_1_output[
    [target, *candidate_predictors, "INCOME", "EDUC4"]
].dropna(
    subset=[target, *candidate_predictors]
).copy()

print("Rows retained:", len(model_df))
print(
    "Rows removed because the response or a candidate predictor was missing:",
    len(tutorial_1_output) - len(model_df),
)

X = model_df[candidate_predictors].astype(float)
y = model_df[target].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=1099,
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

assert set(X_train.index).isdisjoint(set(X_test.index)), (
    "Training and test indices must not overlap."
)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

print("The test set is locked for final evaluation.")


#### <img src="tutorial-icons/human_check.png" alt="Human Check" width="26" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Was the split created before feature selection?
- Are test rows absent from all selection calculations?
- Why is preserving the original index useful for a later social audit?
- Does dropping incomplete rows change the population represented by the model?

### Part 3: Build a Training-Only Cross-Validation Scorer

The scorer must handle:

- a nonempty feature set with `LinearRegression`;
- an empty feature set with a mean-prediction `DummyRegressor`;
- reproducible five-fold cross-validation;
- positive RMSE values.

#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 3

```text
Act as a careful Python tutor.

I already have X_train and y_train.

Write one Jupyter Notebook code cell that:
1. imports LinearRegression, DummyRegressor, KFold, and cross_val_score;
2. defines a function named mean_cv_rmse;
3. accepts features, X_data, y_data, and random_state;
4. uses five folds with shuffle=True;
5. fits LinearRegression when features is nonempty;
6. fits a mean DummyRegressor when features is empty;
7. uses neg_root_mean_squared_error scoring;
8. returns a positive mean RMSE as a float;
9. calculates and prints the null-model training cross-validation RMSE;
10. calculates and prints the full-model training cross-validation RMSE.

Do not use X_test or y_test.
Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Cross-Validation Estimates Selection Performance

Stepwise selection compares many candidate models. Scoring those candidates on the final test set would repeatedly adapt the analysis to that test set. Cross-validation instead rotates held-out folds within the training rows, producing an estimate that can guide selection while preserving the final test rows for one later comparison.

### Deeper explanation

Each observation should act as validation only when every learned operation was fit without it. That includes imputation, scaling, encoding, and feature selection. The fold scores are dependent because training sets overlap, so their standard deviation is a descriptive stability measure rather than a simple independent-sample standard error. Repeated or nested cross-validation can provide a more honest assessment when selection is extensive.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 3: Create the selection scorer
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 3


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

def mean_cv_rmse(
    features,
    X_data,
    y_data,
    random_state=1099,
):
    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state,
    )

    if features:
        model = LinearRegression()
        X_used = X_data[list(features)]
    else:
        model = DummyRegressor(strategy="mean")
        X_used = np.ones((len(X_data), 1))

    rmse_values = -cross_val_score(
        model,
        X_used,
        y_data,
        cv=cv,
        scoring="neg_root_mean_squared_error",
    )

    return float(rmse_values.mean())

null_cv_rmse = mean_cv_rmse(
    [],
    X_train,
    y_train,
)

full_cv_rmse = mean_cv_rmse(
    candidate_predictors,
    X_train,
    y_train,
)

print(f"Null-model training CV RMSE: {null_cv_rmse:.5f}")
print(f"Full-model training CV RMSE: {full_cv_rmse:.5f}")


#### <img src="tutorial-icons/human_check.png" alt="Human Check" width="26" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Does the function ever access the test set?
- Why are scikit-learn’s negative RMSE scores multiplied by $-1$?
- What does the null model predict inside each fold?
- Is a lower cross-validated RMSE preferable?

## 🟠 🛠️ Pólya Step 3 — Carry Out the Plan

**Backbone checkpoint.** Execute in small, observable steps. Read generated code or actions, stay within scope, and compare outputs with the behavior you predicted.

**In this tutorial:** Execute forward and backward selection without using the test set to make selection decisions.

### Part 4: Forward Selection

Forward selection:

1. starts with the null model;
2. evaluates adding each remaining candidate;
3. chooses the addition with the lowest training cross-validation RMSE;
4. keeps the feature only if improvement exceeds the tolerance;
5. repeats until no remaining addition improves the criterion.

Forward selection searches one path. It does not evaluate every possible subset.

#### <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="26" style="vertical-align:middle; margin-right:8px;"> Without Claude

You would need nested loops, careful feature bookkeeping, a stopping rule, a history table, and protection against test-set leakage.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Forward and Backward Search Different Paths

Forward selection starts with no predictors and adds one at a time. Backward selection starts with all candidate predictors and removes one at a time. Because neither method evaluates every possible subset, their paths can end at different models even when they use the same score.

### Deeper explanation

Forward selection can miss a pair of variables that is useful only together, because neither provides the best single-variable improvement. Backward selection can retain one of several redundant variables depending on small data perturbations. Both are greedy algorithms with path dependence. Recording the full path, not only the final subset, makes instability visible and helps distinguish robust signal from arbitrary selection.


#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 4

```text
Act as a careful Python tutor.

I already have:
- X_train;
- y_train;
- candidate_predictors;
- mean_cv_rmse.

Write one Jupyter Notebook code cell that:
1. defines forward_select;
2. accepts X_data, y_data, candidate_features, tolerance=0.0001,
   and random_state=1099;
3. starts with no selected features;
4. records the null-model score as step 0;
5. evaluates adding each remaining feature;
6. selects the addition with the lowest mean CV RMSE;
7. adds it only if RMSE improves by more than tolerance;
8. records step, action, changed_feature, selected_features, and cv_rmse;
9. stops when no addition clears the threshold;
10. returns the selected list and a DataFrame history;
11. runs the function on the training data;
12. displays the history and prints the final selected features.

Do not use the test set.
Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 4: Forward selection
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 4


In [ ]:
def forward_select(
    X_data,
    y_data,
    candidate_features,
    tolerance=0.0001,
    random_state=1099,
):
    selected = []
    remaining = list(candidate_features)

    current_score = mean_cv_rmse(
        selected,
        X_data,
        y_data,
        random_state=random_state,
    )

    history = [{
        "step": 0,
        "action": "start",
        "changed_feature": None,
        "selected_features": tuple(selected),
        "cv_rmse": current_score,
    }]

    step = 0

    while remaining:
        candidate_results = []

        for feature in remaining:
            proposed_features = [*selected, feature]

            score = mean_cv_rmse(
                proposed_features,
                X_data,
                y_data,
                random_state=random_state,
            )

            candidate_results.append((score, feature))

        best_score, best_feature = min(candidate_results)

        if best_score < current_score - tolerance:
            selected.append(best_feature)
            remaining.remove(best_feature)
            current_score = best_score
            step += 1

            history.append({
                "step": step,
                "action": "add",
                "changed_feature": best_feature,
                "selected_features": tuple(selected),
                "cv_rmse": current_score,
            })
        else:
            break

    return selected, pd.DataFrame(history)

forward_features, forward_history = forward_select(
    X_train,
    y_train,
    candidate_predictors,
)

display(forward_history)

print("Forward-selected features:")
print(forward_features)


#### <img src="tutorial-icons/human_check.png" alt="Human Check" width="26" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Does step 0 contain no predictors?
- Which feature entered first?
- Did every accepted step lower training cross-validated RMSE?
- What prevented the procedure from continuing?
- Would a different tolerance change the selected set?

### Part 5: Backward Selection

Backward selection:

1. starts with the full model;
2. evaluates removing each included feature;
3. chooses the removal with the lowest training cross-validation RMSE;
4. removes the feature only if the score improves by more than the tolerance;
5. repeats until no removal improves the criterion.

Because backward selection starts from the full set, it can follow a different path from forward selection.


#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 5

```text
Act as a careful Python tutor.

I already have:
- X_train;
- y_train;
- candidate_predictors;
- mean_cv_rmse.

Write one Jupyter Notebook code cell that:
1. defines backward_select;
2. accepts X_data, y_data, candidate_features, tolerance=0.0001,
   and random_state=1099;
3. starts with every candidate feature;
4. records the full-model score as step 0;
5. evaluates removing each currently selected feature;
6. selects the removal with the lowest mean CV RMSE;
7. removes it only if RMSE improves by more than tolerance;
8. permits evaluation of the null model if only one feature remains;
9. records step, action, changed_feature, selected_features, and cv_rmse;
10. stops when no removal clears the threshold;
11. returns the selected list and a DataFrame history;
12. runs the function, displays the history, and prints the final features.

Do not use the test set.
Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 5: Backward selection
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 5


In [ ]:
def backward_select(
    X_data,
    y_data,
    candidate_features,
    tolerance=0.0001,
    random_state=1099,
):
    selected = list(candidate_features)

    current_score = mean_cv_rmse(
        selected,
        X_data,
        y_data,
        random_state=random_state,
    )

    history = [{
        "step": 0,
        "action": "start",
        "changed_feature": None,
        "selected_features": tuple(selected),
        "cv_rmse": current_score,
    }]

    step = 0

    while selected:
        candidate_results = []

        for feature in selected:
            proposed_features = [
                item for item in selected
                if item != feature
            ]

            score = mean_cv_rmse(
                proposed_features,
                X_data,
                y_data,
                random_state=random_state,
            )

            candidate_results.append((score, feature, proposed_features))

        best_score, removed_feature, best_features = min(
            candidate_results,
            key=lambda item: item[0],
        )

        if best_score < current_score - tolerance:
            selected = best_features
            current_score = best_score
            step += 1

            history.append({
                "step": step,
                "action": "remove",
                "changed_feature": removed_feature,
                "selected_features": tuple(selected),
                "cv_rmse": current_score,
            })
        else:
            break

    return selected, pd.DataFrame(history)

backward_features, backward_history = backward_select(
    X_train,
    y_train,
    candidate_predictors,
)

display(backward_history)

print("Backward-selected features:")
print(backward_features)


#### <img src="tutorial-icons/human_check.png" alt="Human Check" width="26" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Which feature, if any, was removed first?
- Did every accepted removal improve training cross-validated RMSE?
- Did backward selection retain a feature that forward selection never added?
- Why can different search paths end with different subsets?

### Part 6: Evaluate Final Candidate Models on the Untouched Test Set

We will compare:

- null model;
- full model;
- forward-selected model;
- backward-selected model.

The test set is used here for final comparison—not for another round of feature selection.

#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 6

```text
Act as a careful Python tutor.

I already have:
- X_train, X_test, y_train, y_test;
- candidate_predictors;
- forward_features;
- backward_features.

Write one Jupyter Notebook code cell that:
1. imports mean_absolute_error, mean_squared_error, and r2_score;
2. creates candidate_sets for Null, Full, Forward, and Backward;
3. fits a DummyRegressor for the null set and LinearRegression otherwise;
4. fits every model using training data only;
5. predicts the untouched test set;
6. stores each fitted model and prediction;
7. calculates number of predictors, RMSE, MAE, and R-squared;
8. creates a comparison_table sorted by RMSE;
9. displays the table;
10. prints the feature list for every candidate model.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Selection and Final Evaluation Have Different Jobs

Cross-validation chooses among candidates; the untouched test set evaluates the chosen candidates under one common standard. The full and null models are useful reference points: they show whether selection improved on a simple baseline and whether simplification sacrificed meaningful predictive performance.

### Deeper explanation

The validation criterion chooses among candidates; the test set estimates the chosen procedure. These roles cannot be merged without optimism. After test evaluation, changing the model in response converts the test set into additional validation data, so a new independent test set is required for another unbiased final estimate. The object being evaluated includes preprocessing and selection, not only the final regression equation.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 6: Final test-set comparison
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 6


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

candidate_sets = {
    "Null": [],
    "Full": list(candidate_predictors),
    "Forward": list(forward_features),
    "Backward": list(backward_features),
}

fitted_models = {}
prediction_store = {}
comparison_rows = []

for model_name, features in candidate_sets.items():
    if features:
        model = LinearRegression()
        X_train_used = X_train[features]
        X_test_used = X_test[features]
    else:
        model = DummyRegressor(strategy="mean")
        X_train_used = np.ones((len(X_train), 1))
        X_test_used = np.ones((len(X_test), 1))

    model.fit(X_train_used, y_train)
    predictions = model.predict(X_test_used)

    fitted_models[model_name] = model
    prediction_store[model_name] = predictions

    comparison_rows.append({
        "model": model_name,
        "number_of_predictors": len(features),
        "test_RMSE": mean_squared_error(
            y_test,
            predictions,
        ) ** 0.5,
        "test_MAE": mean_absolute_error(
            y_test,
            predictions,
        ),
        "test_R_squared": r2_score(
            y_test,
            predictions,
        ),
    })

comparison_table = (
    pd.DataFrame(comparison_rows)
    .sort_values("test_RMSE")
    .reset_index(drop=True)
)

display(comparison_table)

print("Candidate feature sets:")
for model_name, features in candidate_sets.items():
    print(f"{model_name}: {features}")


#### <img src="tutorial-icons/human_check.png" alt="Human Check" width="26" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Was the test set used only after both selection procedures ended?
- Which model has the lowest test RMSE?
- Are differences practically meaningful or extremely small?
- Did the smaller model sacrifice much predictive performance?
- Would you choose the same model for explanation, prediction, and deployment?
- Why is the test winner still not guaranteed to be best on future data?

## 🟢 🔎 Pólya Step 4 — Look Back

**Backbone checkpoint.** Do not stop at “it ran.” Ask whether the result answers the original problem, what evidence supports it, what failed, and what should change. Domain expertise matters here because a generic checklist cannot know every real-world failure mode.

**In this tutorial:** Compare the paths, final models, test evidence, and stability before choosing a conclusion.

### <img src="tutorial-icons/look_back.png" alt="Look Back" width="30" style="vertical-align:middle; margin-right:8px;"> Part 7: Look Back at the Selection Paths

Create a short written comparison:

```text
Forward selection began with __________ and added __________ first.
It stopped after selecting __________ predictors because __________.

Backward selection began with __________ and removed __________ first.
It stopped with __________ predictors because __________.

The procedures [did / did not] select the same features.

On the untouched test set, the lowest RMSE came from __________.
Compared with the full model, this model used __________ fewer predictors
and changed test RMSE by __________.

This does not prove that the selected variables are causal or uniquely
important because __________.
```


## AI for Social Good: Feature Selection Changes the Intervention

A real-world [food-rescue volunteer-engagement deployment studied by Ryan Shi and collaborators](https://pubsonline.informs.org/doi/10.1287/inte.2025.0265) illustrates why model selection is not only about finding a smaller equation. The deployment connects modeling choices to the full AI-for-social-good pipeline: scoping the problem, building an algorithm, discovering unintended consequences, testing changes in the real world, and continuing to revise the intervention.

Removing or shrinking a variable can:

- make a model easier to explain or cheaper to operate;
- reduce collection of sensitive information;
- remove context that helps explain structural conditions;
- make a remaining proxy more influential;
- shift errors toward particular groups;
- change which people receive an intervention and therefore change the future data used to retrain the model.

So the selected model should be judged on **technical performance plus deployment behavior**.

#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 7: Compare Group Error Before and After Selection

```text
Act as a careful Python tutor.

I already have:
- model_df;
- y_test;
- prediction_store containing Full, Forward, and Backward predictions.

Write one Jupyter Notebook code cell that:
1. creates an audit table for the test-set indices;
2. retains INCOME and EDUC4;
3. adds absolute error for Full, Forward, and Backward models;
4. creates grouped MAE and count tables by INCOME;
5. creates grouped MAE and count tables by EDUC4;
6. displays both;
7. calculates how each selected model's group MAE differs from the full model;
8. prints a warning that descriptive group differences do not establish
   fairness or unfairness.

Do not refit or reselect a model.
Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

> **Social-good principle:** A feature is not “just a variable” once a model changes who receives attention, resources, or intervention.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 7: Selection and group error
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 7


In [ ]:
selection_audit = model_df.loc[
    y_test.index,
    ["INCOME", "EDUC4"],
].copy()

for model_name in ["Full", "Forward", "Backward"]:
    predictions = prediction_store[model_name]

    selection_audit[f"{model_name}_absolute_error"] = np.abs(
        y_test.to_numpy() - predictions
    )

def grouped_selection_audit(group_column):
    grouped = (
        selection_audit
        .groupby(group_column, dropna=False)
        .agg(
            count=("Full_absolute_error", "size"),
            Full_MAE=("Full_absolute_error", "mean"),
            Forward_MAE=("Forward_absolute_error", "mean"),
            Backward_MAE=("Backward_absolute_error", "mean"),
        )
        .reset_index()
        .sort_values(group_column)
    )

    grouped["Forward_minus_Full_MAE"] = (
        grouped["Forward_MAE"] - grouped["Full_MAE"]
    )

    grouped["Backward_minus_Full_MAE"] = (
        grouped["Backward_MAE"] - grouped["Full_MAE"]
    )

    return grouped

income_selection_audit = grouped_selection_audit("INCOME")
education_selection_audit = grouped_selection_audit("EDUC4")

print("Model-selection error comparison by income category:")
display(income_selection_audit)

print("\nModel-selection error comparison by education category:")
display(education_selection_audit)

print(
    "\nWarning: These descriptive summaries do not establish fairness or "
    "unfairness. Group size, sampling variation, measurement, uncertainty, "
    "and the intended use must also be examined."
)


#### Social-Good Reflection

1. Did model simplification change error similarly for every group?
2. Which groups have small sample sizes?
3. Could removing a socioeconomic feature improve overall RMSE while reducing contextual understanding?
4. Is a variable selected because it represents a meaningful mechanism, a proxy, or a correlation?
5. Would the selected model be used for research, screening, resource allocation, or individual decisions?
6. Who bears the cost when the model is wrong?
7. What documentation should accompany the final selected model?
8. What form of human review and appeal would be required in a high-impact use?


## Tutorial 3 Conclusion

Complete the summary:

```text
Forward selection used training-only cross-validated RMSE and selected:
__________.

Backward selection selected:
__________.

The two procedures [agreed / disagreed] because __________.

On the untouched test set, the lowest RMSE came from __________.
Compared with the full model, it used __________ predictors and changed
test RMSE by __________.

The selected variables should not be described as causal or uniquely
important because __________.

The social audit showed __________, but a complete fairness assessment would
also require __________.

For this educational analysis, I would recommend __________ because
__________.
```


## <img src="tutorial-icons/look_back.png" alt="Look Back" width="36" style="vertical-align:middle; margin-right:9px;"> Final Reflection

1. Define forward selection.
2. Define backward selection.
3. Why can they produce different subsets?
4. What criterion controlled additions and removals?
5. Why was the test set locked until selection ended?
6. How did the assigned reading help explain the procedure?
7. What did Claude make easier?
8. Which part of the code required the strongest human verification?
9. Why is the smallest model not always the most responsible model?
10. Which model would you recommend for this educational analysis, and why?

Level 2 extends these questions to stability, information criteria, ridge, lasso, and grouped categorical features.


## <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="36" style="vertical-align:middle; margin-right:9px;"> Level 2 — Optional Deep Dives

> **Challenge ahead:** Complete Level 1 first; this optional route adds a harder application of the same reading.

Level 2 uses the **same assigned readings**. The additional challenge comes from more complex model comparisons, repeated selection, and deeper Claude-assisted auditing—not from a new reading assignment.


### Part 8: Selection Stability

A selected feature set can depend on:

- the cross-validation folds;
- the random seed;
- the tolerance;
- the available sample;
- correlated predictors;
- measurement noise.

A stable method should not produce radically different stories from small, arbitrary changes.

#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 8

```text
Act as a careful Python tutor.

I already have forward_select and backward_select.

Write one Jupyter Notebook code cell that:
1. uses seeds 11, 29, 1099, 2024, and 2026;
2. reruns forward and backward selection for each seed;
3. creates a DataFrame with method, seed, number_selected, and selected_features;
4. displays that table;
5. creates a feature-frequency table showing how often each predictor was
   selected by each method;
6. displays the frequency table;
7. does not use the test set.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 8: Selection stability
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 8


In [ ]:
stability_rows = []
seeds = [11, 29, 1099, 2024, 2026]

for seed in seeds:
    selected_forward, _ = forward_select(
        X_train,
        y_train,
        candidate_predictors,
        random_state=seed,
    )

    selected_backward, _ = backward_select(
        X_train,
        y_train,
        candidate_predictors,
        random_state=seed,
    )

    stability_rows.append({
        "method": "Forward",
        "seed": seed,
        "number_selected": len(selected_forward),
        "selected_features": tuple(selected_forward),
    })

    stability_rows.append({
        "method": "Backward",
        "seed": seed,
        "number_selected": len(selected_backward),
        "selected_features": tuple(selected_backward),
    })

stability_table = pd.DataFrame(stability_rows)

display(stability_table)

frequency_rows = []

for method in ["Forward", "Backward"]:
    method_rows = stability_table[
        stability_table["method"] == method
    ]

    for feature in candidate_predictors:
        count_selected = method_rows["selected_features"].apply(
            lambda selected: feature in selected
        ).sum()

        frequency_rows.append({
            "method": method,
            "feature": feature,
            "selected_count": int(count_selected),
            "out_of": len(method_rows),
            "selection_percent": (
                100 * count_selected / len(method_rows)
            ),
        })

frequency_table = pd.DataFrame(frequency_rows)

display(
    frequency_table.sort_values(
        ["method", "selected_count", "feature"],
        ascending=[True, False, True],
    )
)


#### Stability Reflection

- Which features were selected every time?
- Which appeared only under certain folds?
- Did forward and backward selection differ in stability?
- Would you present an unstable selected set as a definitive scientific finding?
- What additional resampling would strengthen the analysis?


### Part 9: Information Criteria and Adjusted $R^2$

ISLP Section 6.1 discusses criteria that penalize model complexity.

- Adjusted $R^2$ penalizes unnecessary predictors.
- AIC and BIC combine fit with a complexity penalty.
- BIC generally penalizes additional parameters more strongly than AIC.

These criteria answer a related but not identical question to cross-validation.

#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 9

```text
Act as a careful Python tutor.

Using X_train, y_train, and candidate_sets:
1. fit each candidate model on the training set;
2. calculate training RSS;
3. calculate training R-squared;
4. calculate adjusted R-squared;
5. calculate AIC as n*log(RSS/n) + 2*k;
6. calculate BIC as n*log(RSS/n) + log(n)*k;
7. count the intercept in k;
8. handle the null model correctly;
9. create and display a table;
10. explain in a printed note that these are relative comparison criteria,
    not proof of model truth.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 9: Information criteria
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 9


In [ ]:
information_rows = []
n_train = len(y_train)

for model_name, features in candidate_sets.items():
    if features:
        model = LinearRegression()
        X_used = X_train[features]
    else:
        model = DummyRegressor(strategy="mean")
        X_used = np.ones((n_train, 1))

    model.fit(X_used, y_train)
    fitted = model.predict(X_used)

    residuals = y_train.to_numpy() - fitted
    rss = float(np.sum(residuals ** 2))

    tss = float(np.sum(
        (y_train - y_train.mean()) ** 2
    ))

    r_squared = 1 - rss / tss

    k = len(features) + 1

    if n_train > k:
        adjusted_r_squared = (
            1
            - (1 - r_squared)
            * (n_train - 1)
            / (n_train - k)
        )
    else:
        adjusted_r_squared = np.nan

    aic = n_train * np.log(rss / n_train) + 2 * k
    bic = n_train * np.log(rss / n_train) + np.log(n_train) * k

    information_rows.append({
        "model": model_name,
        "predictors": len(features),
        "training_RSS": rss,
        "training_R_squared": r_squared,
        "adjusted_R_squared": adjusted_r_squared,
        "AIC": aic,
        "BIC": bic,
    })

information_table = pd.DataFrame(information_rows)

display(
    information_table.sort_values("BIC")
)

print(
    "AIC, BIC, and adjusted R-squared compare candidate models under "
    "specific assumptions. They do not prove that a model is true, causal, "
    "stable, or socially appropriate."
)


### <img src="tutorial-icons/theory.png" alt="Theory" width="30" style="vertical-align:middle; margin-right:8px;"> Part 10: Ridge and Lasso Extension Using the Same Assigned Reading

Stepwise methods make discrete include/exclude decisions.

- **Ridge regression** shrinks coefficients toward zero but usually keeps every predictor.
- **Lasso regression** can shrink some coefficients exactly to zero.
- Both require tuning a penalty parameter.
- Standardization matters because the penalty depends on coefficient scale.

This deep dive does not replace forward/backward selection. It compares a different model-selection philosophy.

#### <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="26" style="vertical-align:middle; margin-right:8px;"> Claude Coding Task 10

```text
Act as a careful Python tutor.

I already have X_train, X_test, y_train, y_test, and candidate_predictors.

Write one Jupyter Notebook code cell that:
1. imports StandardScaler, Pipeline, RidgeCV, and LassoCV;
2. creates a ridge pipeline using StandardScaler and RidgeCV with a broad
   logarithmic alpha grid;
3. creates a lasso pipeline using StandardScaler and LassoCV with
   cv=5 and random_state=1099;
4. fits both on training data only;
5. predicts the test set;
6. reports selected alpha, RMSE, MAE, and R-squared;
7. creates a lasso coefficient table using the original predictor names;
8. identifies coefficients that are effectively zero;
9. does not use test performance to tune alpha.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 10: Ridge and lasso comparison
# Paste Claude's generated code below this line.
# Read the code before running it.


#### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="26" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 10


In [ ]:
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

alpha_grid = np.logspace(-5, 3, 100)

ridge_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", RidgeCV(alphas=alpha_grid)),
])

lasso_pipeline = Pipeline([
    ("scale", StandardScaler()),
    (
        "model",
        LassoCV(
            alphas=alpha_grid,
            cv=5,
            random_state=1099,
            max_iter=20000,
        ),
    ),
])

regularized_models = {
    "Ridge": ridge_pipeline,
    "Lasso": lasso_pipeline,
}

regularized_rows = []

for model_name, pipeline in regularized_models.items():
    pipeline.fit(
        X_train[candidate_predictors],
        y_train,
    )

    predictions = pipeline.predict(
        X_test[candidate_predictors]
    )

    selected_alpha = pipeline.named_steps["model"].alpha_

    regularized_rows.append({
        "model": model_name,
        "selected_alpha": selected_alpha,
        "test_RMSE": mean_squared_error(
            y_test,
            predictions,
        ) ** 0.5,
        "test_MAE": mean_absolute_error(
            y_test,
            predictions,
        ),
        "test_R_squared": r2_score(
            y_test,
            predictions,
        ),
    })

regularized_table = pd.DataFrame(regularized_rows)

display(regularized_table)

lasso_coefficients = pd.DataFrame({
    "feature": candidate_predictors,
    "standardized_coefficient":
        lasso_pipeline.named_steps["model"].coef_,
})

lasso_coefficients["effectively_zero"] = (
    lasso_coefficients["standardized_coefficient"].abs() < 1e-8
)

display(
    lasso_coefficients.sort_values(
        "standardized_coefficient",
        key=lambda column: column.abs(),
        ascending=False,
    )
)


#### <img src="tutorial-icons/theory.png" alt="Theory" width="26" style="vertical-align:middle; margin-right:8px;"> Deep-Dive Comparison Using the Assigned Reading

Revisit the assigned Sections 6.1 and 6.2 and explain:

1. why ridge normally keeps all predictors;
2. why lasso can produce a sparse model;
3. why coefficient standardization matters;
4. why stepwise and lasso may select different variables;
5. why correlated predictors make “importance” unstable;
6. which method is easier to explain to a nontechnical stakeholder;
7. which method appears strongest for prediction in this example.


### Part 11: Grouped Categorical Features

In Tutorial 2, `INCOME` and `EDUC4` became sets of indicator columns.

Selecting one indicator at a time can create a difficult interpretation: part of a categorical variable may enter while the rest stays out.

A more defensible approach is **grouped selection**:

- add or remove all indicators for `INCOME` together;
- add or remove all indicators for `EDUC4` together;
- compare models at the conceptual-feature level.

#### Claude Deep-Dive Prompt

```text
Act as a model-selection designer.

Explain how to extend the forward and backward procedures so candidate
features can be groups of columns.

Use this structure:
- each numeric predictor is a one-column group;
- INCOME is one group containing all of its indicator columns;
- EDUC4 is one group containing all of its indicator columns.

Provide pseudocode first.
Then identify:
1. how the selected conceptual groups should be stored;
2. how groups expand into design-matrix columns;
3. how to prevent partial category selection;
4. how to report the reference category;
5. one drawback of grouped stepwise selection.

Do not use the test set for group selection.
```


## Sources and Further Reading

- James, Gareth, Daniela Witten, Trevor Hastie, Robert Tibshirani, and Jonathan Taylor. _An Introduction to Statistical Learning_, Sections 6.1–6.2.
- Kearns, Michael, and Aaron Roth. _The Ethical Algorithm_, Chapter 2. **Background social-good reference; not assigned.**
- Zheyuan Ryan Shi, Rayid Ghani, and Fei Fang. [A Deployment Study of a Data-Driven Volunteer Engagement System for Food Security](https://pubsonline.informs.org/doi/10.1287/inte.2025.0265)
- Xavier Bourret Sicotte. [Subset Selection in Python](https://xavierbourretsicotte.github.io/subset_selection.html)
- University of Pittsburgh Health Sciences Library System. [Health-related quality of life measures and social determinants of health in a large U.S. national survey](https://datacatalog.hsls.pitt.edu/dataset/100)
